## 1. Setup and artifact inspection


In [ ]:
%cd /content
![ -d FlakyLens ] || git clone -q https://github.com/UT-SE-Research/FlakyLens.git
%cd /content/FlakyLens
!echo "=== GPU ===" && nvidia-smi --query-gpu=name,memory.total --format=csv
!echo "=== Python / installed libs ===" && python --version && pip list 2>/dev/null | grep -iE "^(torch|transformers|captum|numpy|pandas|scikit-learn) "
!echo "=== models/README.md ===" && cat models/README.md
!echo "=== src (top level) ===" && ls -la src
!echo "=== shell scripts ===" && find . -name "*.sh"
!echo "=== FlakeBench ===" && ls -la FlakeBench
!echo "=== big files ===" && find . -size +5M -not -path "./.git/*" -exec ls -lh {} \;
!echo "=== README run section ===" && grep -n -iE "zenodo|docker load|tar.gz|rq1|models" README.md

/content
/content/FlakyLens
=== GPU ===
name, memory.total [MiB]
Tesla T4, 15360 MiB
=== Python / installed libs ===
Python 3.13.16
numpy                                 2.4.6
pandas                                2.2.3
scikit-learn                          1.6.1
torch                                 2.14.1+cu130
transformers                          5.22.1
=== models/README.md ===
This directory contains the trained models
The trained models are in ```https://utexas.box.com/s/lp1wx8is3ad3vhemql7gp835sc910we8```
=== src (top level) ===
total 344
drwxr-xr-x 5 root root  4096 Nov 23 08:48 .
drwxr-xr-x 8 root root  4096 Nov 23 08:48 ..
drwxr-xr-x 2 root root  4096 Nov 23 08:48 Attributions_scores
-rw-r--r-- 1 root root 24894 Nov 23 08:48 Bert_train_per_project.py
-rw-r--r-- 1 root root  4293 Nov 23 08:48 calculate_most_and_least_imp_tokens.py
-rw-r--r-- 1 root root  3802 Nov 23 08:48 cig.py
-rw-r--r-- 1 root root  3811 Nov 23 08:48 codebert_model.py
-rw-r--r-- 1 root root 22226 Nov 23 08:

In [ ]:
%cd /content/FlakyLens/src
!echo "=== rq1.sh ===" && cat rq1.sh
!echo "=== per_project_prediction.sh ===" && cat per_project_prediction.sh
!echo "=== where Testing_per_project.py loads models ===" && grep -n -iE "from_pretrained|load_state_dict|torch.load|model_path|\.pt\b|\.bin|\.pth|flaky-test-categorization" Testing_per_project.py | head -30

/content/FlakyLens/src
=== rq1.sh ===
#!/bin/bash
currentDir=$(pwd)
bash per_project_prediction.sh FlakyLens "BERT"
cd ../results/scripts
bash per_category_parse_result.sh ../per_Category_Evaluation_BERT-FlakyLens.txt
rm ../per_Category_Evaluation_BERT-FlakyLens.txt
cd $currentDir
=== per_project_prediction.sh ===
#Flakify's flaky test prediction using per-project validation
#dataset=$1
dataset_name=$(echo $1 | cut -d'_' -f1)
data_path="../dataset"
results_path="../models"
#dataset_file="${data_path}/${dataset_name}/${dataset_name}_dataset.csv" 
dataset_file="${data_path}/${dataset_name}/${dataset_name}_dataset_with_nonflaky_indented.csv"  
model_weights="${results_path}/per_project_model_weights_on__dataset"
results_file="${results_path}/sklearn_predictor/per_project_validation_results_on_${dataset}_dataset"

#if [[ ! -d "${results_path}/sklearn_predictor" ]]; then
#    mkdir -p "${results_path}/sklearn_predictor"
#fi
#Flakify_cross_validation_results_on_IDoFT_dataset_Flakify.csv
if [

In [ ]:
import pandas as pd
base = "/content/FlakyLens"
for f in ["FlakeBench/FlakeBench_dataset.csv",
          "src/FlakyLens_Categorization_PerProject-result/Finetuned_Result_with_tokens.csv"]:
    df = pd.read_csv(f"{base}/{f}")
    print(f"\n=== {f} ===  shape={df.shape}")
    print("columns:", list(df.columns))
    print(df.drop(columns=[c for c in df.columns if "code" in c.lower() or "token" in c.lower()], errors="ignore").head(3))



=== FlakeBench/FlakeBench_dataset.csv ===  shape=(8574, 6)
columns: ['id', 'project', 'test_name', 'full_code', 'label', 'category']
   id             project                                          test_name  \
0   1       apache_hadoop  TestDelegationTokenRenewer.testAddRemoveRenewA...   
1   2         neo4j_neo4j  RobustJobSchedulerWrapperTest.shouldBeAbleToCa...   
2   3  eclipse_xtext-core           RequestManagerTest.testRunWriteAfterRead   

         label  category  
0   async wait         0  
1  concurrency         1  
2  concurrency         1  

=== src/FlakyLens_Categorization_PerProject-result/Finetuned_Result_with_tokens.csv ===  shape=(8574, 5)
columns: ['project_group', 'test_code', 'Prediction', 'Ground_Truth', 'Token_List']
   project_group  Prediction  Ground_Truth
0              1           0             0
1              1           1             1
2              1           2             3


## 2. Reproducing the reported metrics from saved predictions

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, f1_score, confusion_matrix
pd.set_option("display.width", 250)

base = "/content/FlakyLens"
ds  = pd.read_csv(f"{base}/FlakeBench/FlakeBench_dataset.csv")
res = pd.read_csv(f"{base}/src/FlakyLens_Categorization_PerProject-result/Finetuned_Result_with_tokens.csv")

# 1. Category number -> name, and class sizes
names = ds.groupby("category")["label"].first().sort_index()
print("=== categories ===");  print(pd.concat([names, ds["category"].value_counts().sort_index()], axis=1, keys=["name","count"]))

y, p = res["Ground_Truth"], res["Prediction"]

# 2. Per-category precision / recall / F1
print("\n=== per-category report ===")
print(classification_report(y, p, target_names=list(names), digits=4))

# 3. Different ways the paper's headline number could be computed
print("Correct predictions:", (y == p).sum(), "of", len(y), "(paper says 8,468 of 8,574)")
print("Macro F1, all 6 classes, all tests pooled :", round(100*f1_score(y, p, average="macro"), 2))
print("Macro F1, 5 flaky classes only, pooled    :", round(100*f1_score(y, p, labels=[0,1,2,3,4], average="macro"), 2))
fold = {g: f1_score(d.Ground_Truth, d.Prediction, average="macro") for g, d in res.groupby("project_group")}
print("Macro F1 per fold (6 classes)             :", {g: round(100*v, 2) for g, v in fold.items()},
      " mean:", round(100*sum(fold.values())/len(fold), 2))

# 4. Confusion matrix: rows = true category, columns = predicted
print("\n=== confusion matrix ===")
print(pd.DataFrame(confusion_matrix(y, p), index=[f"T:{n}" for n in names], columns=[f"P:{n}" for n in names]))

=== categories ===
                           name  count
category                              
0                    async wait     76
1                   concurrency     37
2                          time     33
3         unordered collections     41
4         test order dependency     93
5                     non-flaky   8294

=== per-category report ===
                       precision    recall  f1-score   support

           async wait     0.5495    0.6579    0.5988        76
          concurrency     0.4000    0.3243    0.3582        37
                 time     0.6667    0.7879    0.7222        33
unordered collections     0.6415    0.8293    0.7234        41
test order dependency     0.7794    0.5699    0.6584        93
            non-flaky     1.0000    0.9999    0.9999      8294

             accuracy                         0.9876      8574
            macro avg     0.6728    0.6949    0.6768      8574
         weighted avg     0.9880    0.9876    0.9875      8574

Correct

## 3. Formatting patterns in flaky and non-flaky tests

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
warnings.filterwarnings("ignore"); pd.set_option("display.width", 250)

ds = pd.read_csv("/content/FlakyLens/FlakeBench/FlakeBench_dataset.csv")
code   = ds["full_code"].fillna("").astype(str)
y_bin  = (ds["category"] != 5).astype(int)          # 1 = flaky, 0 = non-flaky
y_cat  = ds["category"]
groups = ds["project"]
gkf    = GroupKFold(n_splits=4)

# ---- Part 1: surface features (nothing about flakiness) ----
def surface(c):
    lines = c.split("\n"); nonblank = [l for l in lines if l.strip()]
    return {"n_chars": len(c), "n_lines": len(lines),
            "has_tab": int("\t" in c), "has_CR": int("\r" in c),
            "frac_blank": (len(lines)-len(nonblank))/max(len(lines),1),
            "n_comments": c.count("//") + c.count("/*"),
            "starts_with_@": int(c.lstrip().startswith("@")),
            "n_@Test": c.count("@Test"),
            "max_indent": max((len(l)-len(l.lstrip()) for l in nonblank), default=0)}
X_surf = pd.DataFrame([surface(c) for c in code])
print("=== Part 1: surface-feature MEANS, non-flaky (0) vs flaky (1) ===")
print(X_surf.groupby(y_bin).mean().T.round(3))

def run(title, make_model, X, y, labels=None):
    preds, fold_f1 = np.empty(len(y), dtype=int), []
    for tr, te in gkf.split(X, y, groups):
        m = make_model(); m.fit(X.iloc[tr], y.iloc[tr]); preds[te] = m.predict(X.iloc[te])
        fold_f1.append(f1_score(y.iloc[te], preds[te], average="macro", labels=labels, zero_division=0))
    print(f"\n=== {title} ===")
    print(classification_report(y, preds, digits=4, zero_division=0))
    print("Macro F1 per fold:", [round(100*f, 2) for f in fold_f1], " mean:", round(100*np.mean(fold_f1), 2))

# ---- Part 2: can surface features alone detect flakiness? ----
run("Part 2: flaky vs non-flaky, SURFACE features, logistic regression",
    lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
    X_surf, y_bin)

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0).fit(X_surf, y_bin)
print("\n=== Part 2b: a depth-3 decision tree on surface features (what does it split on?) ===")
print(export_text(tree, feature_names=list(X_surf.columns)))

# ---- Part 3: TF-IDF word baseline, binary and 6-class ----
tfidf = lambda: make_pipeline(
    TfidfVectorizer(token_pattern=r"[A-Za-z_][A-Za-z0-9_]*", lowercase=False, min_df=2, sublinear_tf=True),
    LogisticRegression(class_weight="balanced", max_iter=3000))
run("Part 3a: flaky vs non-flaky, TF-IDF + logistic regression", tfidf, code, y_bin)
run("Part 3b: 6 categories, TF-IDF + logistic regression (FlakyLens per-fold mean was 65.72)", tfidf, code, y_cat)

=== Part 1: surface-feature MEANS, non-flaky (0) vs flaky (1) ===
category             0         1
n_chars        721.664  1062.179
n_lines         17.191    21.018
has_tab          0.111     0.000
has_CR           0.017     0.000
frac_blank       0.168     0.000
n_comments       0.856     0.075
starts_with_@    0.995     1.000
n_@Test          0.988     1.007
max_indent      10.807     6.714

=== Part 2: flaky vs non-flaky, SURFACE features, logistic regression ===
              precision    recall  f1-score   support

           0     0.9999    0.9959    0.9979      8294
           1     0.8914    0.9964    0.9410       280

    accuracy                         0.9959      8574
   macro avg     0.9456    0.9962    0.9694      8574
weighted avg     0.9963    0.9959    0.9960      8574

Macro F1 per fold: [98.75, 97.6, 94.9, 96.15]  mean: 96.85

=== Part 2b: a depth-3 decision tree on surface features (what does it split on?) ===
|--- frac_blank <= 0.00
|   |--- class: 1
|--- frac_blan

In [ ]:
import pandas as pd, re
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 80)
base = "/content/FlakyLens/FlakeBench"
ds   = pd.read_csv(f"{base}/FlakeBench_dataset.csv")
code = ds["full_code"].fillna("").astype(str)
kind = (ds["category"] != 5).map({True: "flaky", False: "non-flaky"})

# ---- Part 1: strict checks, exact counts ----
inner_blank   = code.apply(lambda c: any(not l.strip() for l in c.strip().split("\n")))  # blank line INSIDE the test
trailing_ws   = code.apply(lambda c: c != c.rstrip())                                    # ends with whitespace/newline
line_comment  = code.str.contains(r"(?m)^\s*(//|/\*|\*)")                                # line that starts as a comment
print("=== blank line inside the test (after trimming ends) ===");  print(pd.crosstab(kind, inner_blank))
print("\n=== code ends with whitespace/newline ===");              print(pd.crosstab(kind, trailing_ws))
print("\n=== has a line starting with a comment ===");            print(pd.crosstab(kind, line_comment))

# ---- Part 2: where did the flaky tests come from? Compare with the FlakyCat file ----
fc = pd.read_csv(f"{base}/Flakicat_dataset_with_modules.csv")
print("\n=== Flakicat_dataset_with_modules.csv ===  shape:", fc.shape)
print("columns:", list(fc.columns))
code_col = [c for c in fc.columns if "code" in c.lower()][0]
fb_flaky_code = set(code[ds["category"] != 5])
fb_non_code   = set(code[ds["category"] == 5])
fc_code       = set(fc[code_col].fillna("").astype(str))
print("FlakeBench flaky tests whose exact code also appears in the FlakyCat file    :", len(fb_flaky_code & fc_code), "of", len(fb_flaky_code))
print("FlakeBench non-flaky tests whose exact code also appears in the FlakyCat file:", len(fb_non_code & fc_code), "of", len(fb_non_code))

# ---- Part 3: one real example of each, from the SAME project, blank lines made visible ----
proj = ds.loc[ds["category"] != 5, "project"].value_counts().index[0]
same = ds[ds["project"] == proj].assign(n=code.str.len())
show = lambda c: "\n".join("<BLANK>" if not l.strip() else l for l in c.strip().split("\n"))
for label, sub in [("FLAKY", same[same.category != 5]), ("NON-FLAKY", same[(same.category == 5) & inner_blank])]:
    row = sub.sort_values("n").iloc[0]
    print(f"\n=== {label} example from project {proj}: {row.test_name} ({row.label}) ===")
    print(show(row.full_code)[:1200])

=== blank line inside the test (after trimming ends) ===
full_code  False  True 
category               
flaky        280      0
non-flaky   3880   4414

=== code ends with whitespace/newline ===
full_code  False  True 
category               
flaky        280      0
non-flaky      0   8294

=== has a line starting with a comment ===
full_code  False  True 
category               
flaky        280      0
non-flaky   6271   2023

=== Flakicat_dataset_with_modules.csv ===  shape: (8574, 7)
columns: ['id', 'project', 'module', 'test_name', 'full_code', 'label', 'category']
FlakeBench flaky tests whose exact code also appears in the FlakyCat file    : 280 of 280
FlakeBench non-flaky tests whose exact code also appears in the FlakyCat file: 8155 of 8155

=== FLAKY example from project wildfly_wildfly: b19048b72669fc0e96665b1b125dc1fda21f5993.testRejectionEAP6 (test order dependency) ===
@Test
public void testRejectionsEAP6() throws Exception {
    testTransformer("subsystem.xml", ModelTestC

### 3.1. Do formatting cues reach the model?


In [ ]:
%cd /content/FlakyLens/src
!echo "=== preprocessing: whitespace / comment / newline handling ===" && grep -n -E "strip\(|replace\(|re\.sub|split\(|comment|\\\\n|max_length|truncat|tokenizer\(|encode_plus|batch_encode" data_processing.py utils.py Testing_per_project.py | head -60

/content/FlakyLens/src
=== preprocessing: whitespace / comment / newline handling ===
data_processing.py:21:    token = token.lower().strip()
data_processing.py:22:    token = re.sub(r'^[ġċ]+', '', token)  # Remove leading artifacts
data_processing.py:23:    token = re.sub(r'[^\w\s]', '', token)  # Remove punctuation
data_processing.py:91:        category_text = output_category.split("Category:")[-1].strip()  # Get everything after "Category:"
data_processing.py:92:        predicted_category = category_text.split()[0]  # Take the first word (e.g., "Async")
data_processing.py:177:    print("\n✅ Top-k Tokens Based on Attention Scores (Only from Test Code):")
data_processing.py:266:#    print("\n✅ Top-k Tokens Based on Attention Scores (Only from Test Code):")
data_processing.py:297:#    print("\n🔹 Tokenized Full Input:")
data_processing.py:300:#    print("\n🔹 Tokenized Test Method:")
data_processing.py:350:#    print("\n✅ Top-k Tokens Based on Attention Scores (Only from Test Code):")
da

In [ ]:
import pandas as pd, re
pd.set_option("display.width", 250)
ds  = pd.read_csv("/content/FlakyLens/FlakeBench/FlakeBench_dataset.csv")
res = pd.read_csv("/content/FlakyLens/src/FlakyLens_Categorization_PerProject-result/Finetuned_Result_with_tokens.csv")

# ---- Part A: what do the model's tokens look like? ----
rk = (res["Ground_Truth"] != 5).map({True: "flaky", False: "non-flaky"})
toks = res["Token_List"].fillna("").astype(str)
for k in ["flaky", "non-flaky"]:
    t = toks[rk == k].iloc[0]
    print(f"=== {k}: first 300 chars of Token_List ===\n{t[:300]!r}\n=== last 150 chars ===\n{t[-150:]!r}\n")

# ---- Part B: do newline / comment markers survive into the tokens? ----
markers = {"newline token 'Ċ'": "Ċ", "double newline 'ĊĊ'": "ĊĊ", "'//' comment": "//", "tab token 'ĉ'": "ĉ", "literal \\n": "\\n"}
print(pd.DataFrame({name: toks.str.contains(m, regex=False).groupby(rk).mean().round(3) for name, m in markers.items()}))

# ---- Part C: does the test name match the method defined in the code? ----
kind = (ds["category"] != 5).map({True: "flaky", False: "non-flaky"})
name_in_code = ds["full_code"].fillna("").str.extract(r"void\s+(\w+)\s*\(")[0]
name_in_list = ds["test_name"].astype(str).str.split(".").str[-1]
match = (name_in_code == name_in_list)
print("\n=== test_name matches the method name in the code? ===");  print(pd.crosstab(kind, match))
print(ds.loc[~match & (kind == "flaky"), ["project", "test_name"]].assign(method_in_code=name_in_code).head(8).to_string())

=== flaky: first 300 chars of Token_List ===
'sleep;Interrupted;renewer;cancelled;Action;Action;renewer;Exception;ancelled;Renew'
=== last 150 chars ===
'sleep;Interrupted;renewer;cancelled;Action;Action;renewer;Exception;ancelled;Renew'

=== non-flaky: first 300 chars of Token_List ===
'Range;Argument;expected;class;new;Exception;IOException;Illegal;Offset;Constructor'
=== last 150 chars ===
'Range;Argument;expected;class;new;Exception;IOException;Illegal;Offset;Constructor'

              newline token 'Ċ'  double newline 'ĊĊ'  '//' comment  tab token 'ĉ'  literal \n
Ground_Truth                                                                                 
flaky                     0.000                0.000           0.0            0.0         0.0
non-flaky                 0.209                0.209           0.0            0.0         0.0

=== test_name matches the method name in the code? ===
col_0      False  True 
category               
flaky          9    271
non-flaky    5

In [ ]:
%cd /content/FlakyLens/src
!echo "=== how test_data is built before tokenizing ===" && sed -n '180,200p' Testing_per_project.py
!echo "=== where Token_List is created ===" && grep -n -B4 -A2 "Token_List" Testing_per_project.py utils.py data_processing.py | head -40

/content/FlakyLens/src
=== how test_data is built before tokenizing ===
    attribution_csvfile_name = attributions_dir+dataset_category+"_attributions_project_group_"+str(project_group)+".csv"
    #print("attribution_csvfile_name=", attribution_csvfile_name)
    #exit()
    # Check if file exists and delete it
    if calculate_attribution and os.path.exists(attribution_csvfile_name):
        os.remove(attribution_csvfile_name)
        #print(f"Deleted existing file: {attribution_csvfile_name}")

    if os.path.exists(attribution_csvfile_name):
        os.remove(attribution_csvfile_name) 
    for g, x_test_chunk in x_test.groupby(np.arange(len(x_test)) // n):
        count +=1
        # Ensure x_test_chunk is a DataFrame
        if isinstance(x_test_chunk, pd.Series):
            x_test_chunk = x_test_chunk.to_frame().T
        # Convert DataFrame to list of strings
        test_data = x_test_chunk.iloc[:, 0].tolist() if len(x_test_chunk) > 1 else [x_test_chunk.iloc[0, 0]]
        test

### 3.2. Additional check: tests sharing a name

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
ds = pd.read_csv("/content/FlakyLens/FlakeBench/FlakeBench_dataset.csv")

fl = ds[ds["category"] != 5].copy()
fl["base_name"] = fl["test_name"].astype(str).str.replace(r"(\.\d+)+\.?$", "", regex=True)   # strip ".2", ".2." suffixes
dups = fl[fl.duplicated("base_name", keep=False)]
print("Flaky rows that share a test name with another flaky row:", len(dups), "rows,", dups["base_name"].nunique(), "distinct tests")
print(dups.groupby("base_name").agg(rows=("id", "size"),
                                     labels=("label", lambda s: sorted(set(s))),
                                     identical_code=("full_code", lambda s: s.nunique() == 1)).to_string())

Flaky rows that share a test name with another flaky row: 10 rows, 5 distinct tests
                                                               rows                   labels  identical_code
base_name                                                                                                   
13d9053050c9e4fb2ef049db6a37d3f6eebf48fa.testProcessAction_ok     2  [test order dependency]           False
13d9053050c9e4fb2ef049db6a37d3f6eebf48fa.testRender_ok            2  [test order dependency]           False
b19048b72669fc0e96665b1b125dc1fda21f5993.testBind                 2  [test order dependency]           False
b19048b72669fc0e96665b1b125dc1fda21f5993.testListBindings         2  [test order dependency]           False
b19048b72669fc0e96665b1b125dc1fda21f5993.testRebind               2  [test order dependency]           False


## 4. Obtaining the model checkpoints

In [ ]:
%%bash
cd /content/FlakyLens/models
SHARED=lp1wx8is3ad3vhemql7gp835sc910we8
for ID in 1968605735600 1968617291955 1968610866031 1968617178565; do
  echo "--- downloading file $ID ---"
  curl -L -s -J -O "https://utexas.app.box.com/index.php?rm=box_download_shared_file&shared_name=${SHARED}&file_id=f_${ID}"
done
echo "=== what we got ==="
ls -lh
file *.pt 2>/dev/null || echo "No .pt files saved - see the listing above"
echo "=== where token_list is built ==="
grep -n "token_list" /content/FlakyLens/src/Testing_per_project.py | head -15

--- downloading file 1968605735600 ---
--- downloading file 1968617291955 ---
--- downloading file 1968610866031 ---
--- downloading file 1968617178565 ---
=== what we got ===
total 20K
-rw-r--r-- 1 root root 2.0K Nov 23 09:07 per_project_model_weights_on__dataset_project_group_1.pt
-rw-r--r-- 1 root root 2.0K Nov 23 09:07 per_project_model_weights_on__dataset_project_group_2.pt
-rw-r--r-- 1 root root 2.0K Nov 23 09:07 per_project_model_weights_on__dataset_project_group_3.pt
-rw-r--r-- 1 root root 2.0K Nov 23 09:07 per_project_model_weights_on__dataset_project_group_4.pt
-rw-r--r-- 1 root root  133 Nov 23 08:48 README.md
per_project_model_weights_on__dataset_project_group_1.pt: JavaScript source, ASCII text, with very long lines (1787)
per_project_model_weights_on__dataset_project_group_2.pt: JavaScript source, ASCII text, with very long lines (1787)
per_project_model_weights_on__dataset_project_group_3.pt: JavaScript source, ASCII text, with very long lines (1787)
per_project_model_we

In [ ]:
import requests, json
# 1. What files does the Zenodo record contain, and how big are they?
r = requests.get("https://zenodo.org/api/records/15761937", timeout=60).json()
print("=== Zenodo record:", r.get("metadata", {}).get("title"), "===")
for f in r.get("files", []):
    print(f"{f['key']:60s} {f['size']/1e9:6.2f} GB   {f['links']['self']}")

# 2. What did Box actually send us? (first 600 characters)
print("\n=== start of the 2 KB 'model' file from Box ===")
print(open("/content/FlakyLens/models/per_project_model_weights_on__dataset_project_group_1.pt").read()[:600])

=== Zenodo record: Understanding and Improving Flaky Test Classification ===
README.md                                                      0.00 GB   https://zenodo.org/api/records/15761937/files/README.md/content
flakylens_latest.tar.gz                                       10.98 GB   https://zenodo.org/api/records/15761937/files/flakylens_latest.tar.gz/content

=== start of the 2 KB 'model' file from Box ===

 <!DOCTYPE html><html lang="en-US"><head><meta name="robots" content="noindex, nofollow"><title>Box - Free Online File Storage, Internet File Sharing, Access Documents &amp; Files Anywhere, Backup Data, Share Files</title><link rel="shortcut icon" href="/favicon.ico" type="image/x-icon" /><link href="https://cdn01.boxcdn.net/_assets/css/transition/style_not_found-pwZoby.css" rel="stylesheet" type="text/css" media="screen" /></head><body>   <div class="center"> <a href="https://utexas.app.box.com/" class="logo" ><img src="https://cdn01.boxcdn.net/_assets/img/not_found_box_logo-Cz

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
%%bash
cd /content/FlakyLens/models
rm -f *.pt                                   # remove any leftovers from earlier attempts
echo "=== found in Drive ==="
find /content/drive/MyDrive -maxdepth 3 -name "per_project_model_weights_on__dataset_project_group_*.pt" -exec ls -lh {} \;
find /content/drive/MyDrive -maxdepth 3 -name "per_project_model_weights_on__dataset_project_group_*.pt" -exec cp {} . \;
echo "=== copied into models/ ==="
ls -lh
file *.pt

=== found in Drive ===
-rw------- 1 root root 478M Nov 23 10:13 /content/drive/MyDrive/flakylens_models/per_project_model_weights_on__dataset_project_group_1.pt
-rw------- 1 root root 478M Nov 23 10:13 /content/drive/MyDrive/flakylens_models/per_project_model_weights_on__dataset_project_group_2.pt
-rw------- 1 root root 478M Nov 23 10:14 /content/drive/MyDrive/flakylens_models/per_project_model_weights_on__dataset_project_group_4.pt
-rw------- 1 root root 478M Nov 23 10:14 /content/drive/MyDrive/flakylens_models/per_project_model_weights_on__dataset_project_group_3.pt
=== copied into models/ ===
total 1.9G
-rw------- 1 root root 478M Nov 23 10:23 per_project_model_weights_on__dataset_project_group_1.pt
-rw------- 1 root root 478M Nov 23 10:23 per_project_model_weights_on__dataset_project_group_2.pt
-rw------- 1 root root 478M Nov 23 10:23 per_project_model_weights_on__dataset_project_group_3.pt
-rw------- 1 root root 478M Nov 23 10:23 per_project_model_weights_on__dataset_project_group

In [ ]:
import torch
sd = torch.load("/content/FlakyLens/models/per_project_model_weights_on__dataset_project_group_1.pt", map_location="cpu")
print("type:", type(sd))
if isinstance(sd, dict):
    keys = list(sd.keys())
    print(len(keys), "tensors;", round(sum(v.numel() for v in sd.values() if hasattr(v, "numel"))/1e6, 1), "M parameters")
    print("first keys:", keys[:4])
    print("last keys :", [(k, tuple(sd[k].shape)) for k in keys[-6:]])

type: <class 'collections.OrderedDict'>
203 tensors; 125.0 M parameters
first keys: ['bert.embeddings.word_embeddings.weight', 'bert.embeddings.position_embeddings.weight', 'bert.embeddings.token_type_embeddings.weight', 'bert.embeddings.LayerNorm.weight']
last keys : [('bert.pooler.dense.weight', (768, 768)), ('bert.pooler.dense.bias', (768,)), ('fc1.weight', (512, 768)), ('fc1.bias', (512,)), ('fc2.weight', (6, 512)), ('fc2.bias', (6,))]


## 5. Reproducing FlakyLens predictions on fold 1


In [ ]:
%cd /content/FlakyLens/src
!echo "=== which pretrained model / tokenizer? ===" && grep -n -E "codebert|model_name\s*=|from_pretrained" Testing_per_project.py utils.py codebert_model.py | head -15
!echo "=== rows per fold in the authors' results ===" && python -c "import pandas as pd; print(pd.read_csv('FlakyLens_Categorization_PerProject-result/Finetuned_Result_with_tokens.csv')['project_group'].value_counts().sort_index().to_string())"

/content/FlakyLens/src
=== which pretrained model / tokenizer? ===
Testing_per_project.py:33:from codebert_model import BERT_Arch
Testing_per_project.py:36:from codebert_model import BERT_Arch
Testing_per_project.py:37:from utils import set_seed, setup_logging, seed_worker, train, evaluate, parse_cr, codebert_model_define, init_setup
Testing_per_project.py:381:    model_name, tokenizer, auto_model = codebert_model_define()
Testing_per_project.py:386:    prediction_time_codebert = 0
utils.py:712:def codebert_model_define():	
utils.py:713:    model_name = "microsoft/codebert-base"
utils.py:714:    model_config = AutoConfig.from_pretrained(model_name, return_dict=False, output_hidden_states=True)
utils.py:715:    tokenizer = AutoTokenizer.from_pretrained(model_name)
utils.py:716:    auto_model = AutoModel.from_pretrained(model_name, config=model_config)	
utils.py:720:    model_name = "Salesforce/codet5-small"
utils.py:721:    tokenizer = RobertaTokenizer.from_pretrained(model_name)
utils.

In [ ]:
import sys, inspect, importlib
sys.path.insert(0, "/content/FlakyLens/src")
print(open("/content/FlakyLens/src/codebert_model.py").read())
try:
    cm = importlib.import_module("codebert_model")
    for name, cls in inspect.getmembers(cm, inspect.isclass):
        if cls.__module__ == "codebert_model":
            print("CLASS:", name, inspect.signature(cls.__init__))
except Exception as e:
    print("IMPORT FAILED:", type(e).__name__, e)

import torch
from torch import nn
from transformers import AutoModel

# set up the neural network for CodeBERT fine-tuning
class BERT_Arch(nn.Module):
    def __init__(self, auto_model, output_layer):
        super(BERT_Arch, self).__init__()
        self.bert = auto_model
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU() #For activation
        #self.fc1 = nn.Linear(2*768, 512)
        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, output_layer)
        self.softmax = nn.LogSoftmax(dim=-1)

         # Ensure all parameters require gradients
        for param in self.parameters():
            param.requires_grad = True


    # define the forward pass
    def forward(self, sent_id, mask, is_embedding=False):
        if is_embedding:
            # If sent_id is actually embeddings
            embeddings = sent_id
            chunk_size = 512  # For chunking
            total_seq_length = embeddings.size(1)
            cls_hs = []
            for start

In [ ]:
import sys, time, torch, numpy as np, pandas as pd
from transformers import AutoConfig, AutoTokenizer, AutoModel
from sklearn.metrics import f1_score
sys.path.insert(0, "/content/FlakyLens/src")
from codebert_model import BERT_Arch          # the authors' own model class

device, MODEL_NAME = "cuda", "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_flakylens(group):
    """Build BERT_Arch exactly as utils.codebert_model_define() does, then load the fold's trained weights."""
    config = AutoConfig.from_pretrained(MODEL_NAME, return_dict=False, output_hidden_states=True)
    model  = BERT_Arch(AutoModel.from_pretrained(MODEL_NAME, config=config), 6)
    sd = torch.load(f"/content/FlakyLens/models/per_project_model_weights_on__dataset_project_group_{group}.pt", map_location="cpu")
    r = model.load_state_dict(sd, strict=False)
    print(f"fold {group}: missing keys = {r.missing_keys} | unexpected keys = {r.unexpected_keys}")
    return model.to(device).eval()             # eval() switches dropout OFF

@torch.no_grad()                               # prediction only: no gradients needed
def predict(model, codes, batch_size=32):
    probs = []
    for i in range(0, len(codes), batch_size):
        enc = tokenizer(list(codes[i:i+batch_size]), max_length=512, padding="max_length",
                        truncation=True, return_tensors="pt")      # same settings as their batch_encode_plus call
        out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))  # log-probabilities
        probs.append(out.exp().cpu())
    probs = torch.cat(probs).numpy()
    return probs.argmax(1), probs

res  = pd.read_csv("/content/FlakyLens/src/FlakyLens_Categorization_PerProject-result/Finetuned_Result_with_tokens.csv")
fold = res[res["project_group"] == 1].reset_index(drop=True)

model1 = load_flakylens(1)
t0 = time.time()
our_pred, our_prob = predict(model1, fold["test_code"].astype(str).tolist())
print(f"Predicted {len(fold)} tests in {time.time()-t0:.0f} s")

agree = our_pred == fold["Prediction"].values
print("Agreement with the authors' saved predictions:", agree.sum(), "of", len(fold))
print("Macro F1, fold 1 - ours:", round(100*f1_score(fold["Ground_Truth"], our_pred, average="macro"), 2),
      "| authors' saved: 63.2")
if (~agree).any():
    print(pd.crosstab(fold["Prediction"].values[~agree], our_pred[~agree], rownames=["authors"], colnames=["ours"]))

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fold 1: missing keys = [] | unexpected keys = []
Predicted 2229 tests in 64 s
Agreement with the authors' saved predictions: 2229 of 2229
Macro F1, fold 1 - ours: 63.2 | authors' saved: 63.2


## 6. Formatting-only input perturbations

In [ ]:
import pandas as pd, numpy as np
pd.set_option("display.width", 250)

# ---- formatting-only transformations (Java behavior unchanged) ----
def _split_tail(c):                     # separate the code from its trailing whitespace
    core = c.rstrip(); return core, c[len(core):]
def strip_trailing(c):   return c.rstrip()
def remove_blank_lines(c):
    core, tail = _split_tail(c); return "\n".join(l for l in core.split("\n") if l.strip()) + tail
def remove_comment_lines(c):            # whole-line // comments and /* ... */ blocks only
    core, tail = _split_tail(c); out, in_block = [], False
    for l in core.split("\n"):
        s = l.strip()
        if in_block:
            if "*/" in s: in_block = False
            continue
        if s.startswith("//"): continue
        if s.startswith("/*"):
            in_block = "*/" not in s; continue
        out.append(l)
    return "\n".join(out) + tail
def flaky_style(c):      return strip_trailing(remove_blank_lines(remove_comment_lines(c)))
def add_trailing_nl(c):  return c + "\n"
def add_blank_line(c):
    lines = c.split("\n")
    i = next((k for k, l in enumerate(lines) if l.rstrip().endswith("{")), 0)
    return "\n".join(lines[:i+1] + [""] + lines[i+1:])

names = {0:"async", 1:"concurrency", 2:"time", 3:"unordered", 4:"order-dep", 5:"non-flaky"}
nf = fold[fold["Ground_Truth"] == 5]["test_code"].astype(str).tolist()
fl = fold[fold["Ground_Truth"] != 5]
fl_code, fl_true = fl["test_code"].astype(str).tolist(), fl["Ground_Truth"].values

# ---- A: non-flaky tests made to look 'flaky-style' ----
rows = []
for label, f in [("original", lambda c: c), ("A1 strip trailing newline", strip_trailing),
                 ("A2 remove blank lines", remove_blank_lines), ("A3 remove comment lines", remove_comment_lines),
                 ("A4 all three", flaky_style)]:
    new = [f(c) for c in nf]; changed = np.array([n != o for n, o in zip(new, nf)])
    pred = predict(model1, new)[0]; to_flaky = pred != 5
    rows.append({"variant": label, "tests changed": int(changed.sum()), "of": len(nf),
                 "% now predicted FLAKY (all)": round(100*to_flaky.mean(), 2),
                 "% predicted FLAKY (among changed)": round(100*to_flaky[changed].mean(), 2) if changed.any() else None,
                 "predicted as": dict(pd.Series(pred[to_flaky]).map(names).value_counts())})
print("=== A: NON-FLAKY tests (fold 1) ===");  print(pd.DataFrame(rows).to_string(index=False))

# ---- B: flaky tests made to look 'non-flaky-style' ----
rows = []
for label, f in [("original", lambda c: c), ("B1 add trailing newline", add_trailing_nl),
                 ("B2 add one blank line", add_blank_line), ("B3 both", lambda c: add_trailing_nl(add_blank_line(c)))]:
    pred = predict(model1, [f(c) for c in fl_code])[0]
    rows.append({"variant": label, "n": len(fl_code),
                 "% now predicted NON-FLAKY": round(100*(pred == 5).mean(), 2),
                 "% correct category": round(100*(pred == fl_true).mean(), 2)})
print("\n=== B: FLAKY tests (fold 1) ===");  print(pd.DataFrame(rows).to_string(index=False))

# ---- one example, so you can check the transformation is formatting-only ----
ex = next(c for c in nf if remove_blank_lines(c) != c and remove_comment_lines(c) != c)
show = lambda c: c.replace("\n\n", "\n<BLANK>\n")
print("\n=== example non-flaky test BEFORE ===\n" + show(ex)[:700] + "\n=== AFTER A4 ===\n" + flaky_style(ex)[:700])

=== A: NON-FLAKY tests (fold 1) ===
                  variant  tests changed   of  % now predicted FLAKY (all)  % predicted FLAKY (among changed)     predicted as
                 original              0 2164                         0.00                                NaN               {}
A1 strip trailing newline           2164 2164                         0.00                               0.00               {}
    A2 remove blank lines           1260 2164                         0.00                               0.00               {}
  A3 remove comment lines            555 2164                         0.00                               0.00               {}
             A4 all three           2164 2164                         0.32                               0.32 {'order-dep': 7}

=== B: FLAKY tests (fold 1) ===
                variant  n  % now predicted NON-FLAKY  % correct category
               original 65                       0.00               55.38
B1 add trailing newli

In [ ]:
import gc, torch, pandas as pd, numpy as np
fold_rows, per_test = [], []
for g in [1, 2, 3, 4]:
    m  = model1 if g == 1 else load_flakylens(g)
    d  = res[res["project_group"] == g]
    nf_code = d[d["Ground_Truth"] == 5]["test_code"].astype(str).tolist()
    f  = d[d["Ground_Truth"] != 5]
    fc, ft = f["test_code"].astype(str).tolist(), f["Ground_Truth"].values

    p_nf_a4 = predict(m, [flaky_style(c) for c in nf_code])[0]       # non-flaky made flaky-style
    p_fl    = predict(m, fc)[0]                                       # flaky, original
    p_fl_b1 = predict(m, [add_trailing_nl(c) for c in fc])[0]        # flaky + trailing newline
    fold_rows.append({"fold": g, "non-flaky n": len(nf_code), "A4: % non-flaky -> flaky": round(100*(p_nf_a4 != 5).mean(), 2),
                      "flaky n": len(fc), "B1: % flaky -> non-flaky": round(100*(p_fl_b1 == 5).mean(), 2),
                      "category acc before": round(100*(p_fl == ft).mean(), 2), "category acc after B1": round(100*(p_fl_b1 == ft).mean(), 2)})
    for c, t, a, b in zip(fc, ft, p_fl, p_fl_b1):
        per_test.append({"fold": g, "true": names[t], "n_tokens": len(tokenizer(c)["input_ids"]),
                         "orig_pred_nonflaky": a == 5, "flipped_to_nonflaky": b == 5})
    if g != 1:
        del m; gc.collect(); torch.cuda.empty_cache()

print("=== per fold ===");  print(pd.DataFrame(fold_rows).to_string(index=False))
pt = pd.DataFrame(per_test)
print(f"\nALL FOLDS: {pt.flipped_to_nonflaky.sum()} of {len(pt)} flaky tests flip to non-flaky after adding one trailing newline "
      f"({100*pt.flipped_to_nonflaky.mean():.1f}%); originally predicted non-flaky: {pt.orig_pred_nonflaky.sum()}")

print("\n=== by true category ===")
print(pt.groupby("true")["flipped_to_nonflaky"].agg(n="size", flipped="sum", pct=lambda s: round(100*s.mean(), 1)).to_string())

print("\n=== mechanism check: by input length (model only sees the first 512 tokens) ===")
pt["length"] = pd.cut(pt["n_tokens"], [0, 256, 512, 10**6], labels=["<=256 tokens", "257-512 tokens", ">512 (truncated: newline never seen)"])
print(pt.groupby("length", observed=False)["flipped_to_nonflaky"].agg(n="size", flipped="sum", pct=lambda s: round(100*s.mean(), 1)).to_string())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (732 > 512). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fold 2: missing keys = [] | unexpected keys = []


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fold 3: missing keys = [] | unexpected keys = []


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fold 4: missing keys = [] | unexpected keys = []
=== per fold ===
 fold  non-flaky n  A4: % non-flaky -> flaky  flaky n  B1: % flaky -> non-flaky  category acc before  category acc after B1
    1         2164                      0.32       65                     35.38                55.38                  33.85
    2         2078                      1.15      103                     13.59                66.02                  51.46
    3         2141                      0.33       58                     13.79                67.24                  60.34
    4         1911                      0.21       54                     20.37                59.26                  48.15

ALL FOLDS: 56 of 280 flaky tests flip to non-flaky after adding one trailing newline (20.0%); originally predicted non-flaky: 0

=== by true category ===
              n  flipped   pct
true                          
async        76        2   2.6
concurrency  37        1   2.7
order-dep    93       42  45.2
time

In [ ]:
import os, pandas as pd

# ---- Part 1: is order-dependency fragility just about length? ----
print("=== median input length (tokens) per category ===")
print(pt.groupby("true")["n_tokens"].median().sort_values().to_string())
print("\n=== flipped / total, by category AND length ===")
print(pt.groupby(["true", "length"], observed=False)["flipped_to_nonflaky"]
        .agg(lambda s: f"{int(s.sum())}/{len(s)}").unstack().to_string())

# ---- Part 2: save results to Google Drive ----
out = "/content/drive/MyDrive/flakylens_results"
os.makedirs(out, exist_ok=True)
pt.to_csv(f"{out}/step16_per_test_trailing_newline.csv", index=False)
pd.DataFrame(fold_rows).to_csv(f"{out}/step16_per_fold.csv", index=False)
print("\nSaved:", os.listdir(out))

=== median input length (tokens) per category ===
true
order-dep      163.0
unordered      243.0
time           294.0
async          438.5
concurrency    442.0

=== flipped / total, by category AND length ===
length      <=256 tokens 257-512 tokens >512 (truncated: newline never seen)
true                                                                        
async               0/22           2/23                                 0/31
concurrency          0/5           1/18                                 0/14
order-dep          39/68           3/20                                  0/5
time                3/12           0/15                                  0/6
unordered           7/21           1/15                                  0/5

Saved: ['step16_per_test_trailing_newline.csv', 'step16_per_fold.csv']


## 7. Cross-checking FlakeBench labels with IDoFT

In [ ]:
%cd /content
![ -d idoft ] || git clone -q --depth 1 https://github.com/TestingResearchIllinois/idoft.git
!echo "=== IDoFT files ===" && ls idoft && wc -l idoft/*.csv
!echo "=== pr-data.csv first lines ===" && head -3 idoft/pr-data.csv
!echo "=== FlakeBench mapping file ===" && wc -l /content/FlakyLens/FlakeBench/filtered_tests_with_owner_sha.csv && head -5 /content/FlakyLens/FlakeBench/filtered_tests_with_owner_sha.csv

/content
=== IDoFT files ===
auto-check-deletion	     auto-run-nondex	  readme.md
auto-check-fix-commit	     auto-update-dataset  repo-info
auto-check-moved-or-rename   format_checker	  tic-fic-data.csv
auto_check_unmaintained      gr-data.csv	  tso-iso-rates.csv
auto-filter-tests-by-letter  odr-tests.csv	  verify_test_type
auto_find_deletion_commit    pr-data.csv
auto-run-dockerized-nondex   py-data.csv
   1030 idoft/gr-data.csv
   1909 idoft/odr-tests.csv
   8076 idoft/pr-data.csv
   1619 idoft/py-data.csv
    246 idoft/tic-fic-data.csv
    102 idoft/tso-iso-rates.csv
  12982 total
=== pr-data.csv first lines ===
Project URL,SHA Detected,Module Path,Fully-Qualified Test Name (packageName.ClassName.methodName),Category,Status,PR Link,Notes
https://github.com/abel533/Mapper,1764748eedb2f320a0d1c43cb4f928c4ccb1f2f5,core,tk.mybatis.mapper.annotation.IdTest.testCompositeKeys,ID,Opened,https://github.com/abel533/Mapper/pull/896,Accepted in the PR https://github.com/abel533/Mapper/pull/666 

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70)
idoft = pd.read_csv("/content/idoft/pr-data.csv")
print("IDoFT shape:", idoft.shape); print("columns:", list(idoft.columns))
for col in [c for c in idoft.columns if c.lower() in ("category", "status")]:
    print(f"\n=== {col} (top 15) ===");  print(idoft[col].value_counts().head(15).to_string())

IDoFT shape: (8075, 8)
columns: ['Project URL', 'SHA Detected', 'Module Path', 'Fully-Qualified Test Name (packageName.ClassName.methodName)', 'Category', 'Status', 'PR Link', 'Notes']

=== Category (top 15) ===
Category
ID             5191
OD             1074
NOD             664
OD-Vic          470
NIO             198
UD              158
TD              129
NDOD;TD          32
OD-Brit          29
NDOD             16
NIO;OD-Vic       14
NDOD;NOD         14
NDOD;OD-Vic       9
NDOI              8
ID;NIO            7

=== Status (top 15) ===
Status
Accepted            2303
Opened              1508
Deleted              459
DeveloperFixed       340
DeveloperWontFix     334
Unmaintained         304
MovedOrRenamed       245
RepoArchived         204
InspiredAFix         162
Rejected             105
MovedToGradle         93
Deprecated            19
Skipped               14
RepoDeleted            8
RepoRenamed            8


In [ ]:
import pandas as pd, re
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
ds    = pd.read_csv("/content/FlakyLens/FlakeBench/FlakeBench_dataset.csv")
idoft = pd.read_csv("/content/idoft/pr-data.csv")
FQN   = "Fully-Qualified Test Name (packageName.ClassName.methodName)"
key   = lambda s: str(s).lower().replace("/", "_")          # 'apache/hadoop' -> 'apache_hadoop'
J     = lambda s: ";".join(sorted(set(s.astype(str))))

# IDoFT: project key, Class.method and method
idoft["proj"] = idoft["Project URL"].astype(str).str.rstrip("/").str.split("github.com/").str[-1].map(key)
nm = idoft[FQN].astype(str).str.replace(r"\[.*\]$", "", regex=True)          # drop parameter suffixes like [1]
idoft["cls_method"] = nm.str.split(".").str[-2:].str.join(".")
idoft["method"]     = nm.str.split(".").str[-1]

# ---- Q1: provenance via commit SHAs ----
m = pd.read_csv("/content/FlakyLens/FlakeBench/filtered_tests_with_owner_sha.csv", header=None, names=["repo", "shas"])
m["proj"] = m["repo"].map(key)
print("Q1. mapping-file projects that match FlakeBench project names:", m["proj"].isin(set(ds["project"].str.lower())).sum(), "of", len(m))
sh = m.assign(sha=m["shas"].astype(str).str.split(";")).explode("sha")
pairs = set(zip(idoft["proj"], idoft["SHA Detected"].astype(str)))
sh["in_idoft"] = [(p, s) in pairs for p, s in zip(sh["proj"], sh["sha"])]
print("    FlakeBench (project, SHA) pairs that are IDoFT detection SHAs:", sh["in_idoft"].sum(), "of", len(sh))

# FlakeBench: project key, Class.method (or method only when the name is 'SHA.method')
ds["proj"] = ds["project"].str.lower()
tn    = ds["test_name"].astype(str).str.replace(r"(\.\d+)+\.?$", "", regex=True)
parts = tn.str.split(".")
has_sha = parts.str[0].str.fullmatch(r"[0-9a-f]{40}").fillna(False)
ds["method"]     = parts.str[-1]
ds["cls_method"] = parts.str[-2:].str.join(".").where(~has_sha)

by_cm = idoft.groupby(["proj", "cls_method"]).agg(cat=("Category", J), status=("Status", J))
by_m  = idoft.groupby(["proj", "method"]).agg(cat=("Category", J), status=("Status", J))
def look(r, col):
    tbl, k = (by_cm, (r.proj, r.cls_method)) if pd.notna(r.cls_method) else (by_m, (r.proj, r.method))
    return tbl[col].get(k)
ds["idoft_cat"]    = ds.apply(lambda r: look(r, "cat"), axis=1)
ds["idoft_status"] = ds.apply(lambda r: look(r, "status"), axis=1)

def simp(c):                                   # collapse IDoFT codes into comparable groups
    if c is None or pd.isna(c): return "not in IDoFT"
    t, out = set(c.split(";")), []
    if t & {"OD", "OD-Vic", "OD-Brit", "NDOD"}: out.append("OD")
    if "ID" in t: out.append("ID")
    if t & {"NOD", "TD", "NDOI"}: out.append("NOD/TD")
    if t & {"NIO", "UD"}: out.append("NIO/UD")
    return "+".join(out) or c
ds["idoft_group"] = ds["idoft_cat"].map(simp)
flaky, nonfl = ds[ds["category"] != 5], ds[ds["category"] == 5]

# ---- Q2: category agreement for flaky tests ----
print("\nQ2. FlakeBench flaky label (rows) vs IDoFT category (columns)")
print(pd.crosstab(flaky["label"], flaky["idoft_group"], margins=True).to_string())
print("    (matched by method name only, because the name had a SHA prefix:", int(has_sha[ds["category"] != 5].sum()), "flaky tests)")

# ---- Q3: FlakeBench NON-flaky tests listed as flaky in IDoFT ----
hit = nonfl[nonfl["idoft_group"] != "not in IDoFT"]
print(f"\nQ3. FlakeBench NON-FLAKY tests found in IDoFT as flaky: {len(hit)} of {len(nonfl)}")
print(hit["idoft_group"].value_counts().to_string())
print(hit[["project", "test_name", "idoft_cat", "idoft_status"]].head(12).to_string(index=False))

Q1. mapping-file projects that match FlakeBench project names: 97 of 98
    FlakeBench (project, SHA) pairs that are IDoFT detection SHAs: 91 of 245

Q2. FlakeBench flaky label (rows) vs IDoFT category (columns)
idoft_group            ID  ID+NIO/UD  OD  not in IDoFT  All
label                                                      
async wait              0          0   0            76   76
concurrency             0          0   0            37   37
test order dependency   0          0  61            32   93
time                    0          0   0            33   33
unordered collections  34          3   0             4   41
All                    34          3  61           182  280
    (matched by method name only, because the name had a SHA prefix: 64 flaky tests)

Q3. FlakeBench NON-FLAKY tests found in IDoFT as flaky: 129 of 8294
idoft_group
OD           91
ID           27
NOD/TD        5
OD+NOD/TD     2
NIO/UD        2
OD+NIO/UD     1
ID+NIO/UD     1
      project                 

### 7.1. Commit-level provenance check

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70)
fb_shas = sh.groupby("proj")["sha"].apply(set).to_dict()          # commits FlakeBench used, per project

h = hit[["proj", "project", "test_name", "cls_method"]].merge(
        idoft[["proj", "cls_method", "SHA Detected", "Module Path", FQN, "Category", "Status"]],
        on=["proj", "cls_method"], how="left")
h["same_commit"] = [str(s) in fb_shas.get(p, set()) for p, s in zip(h["proj"], h["SHA Detected"])]

# ---- (a) version check, one row per conflicting test ----
per = h.groupby(["project", "test_name"]).agg(same_commit=("same_commit", "any"),
                                              cats=("Category", J), status=("Status", J)).reset_index()
per["group"] = per["cats"].map(simp)
print("=== (a) conflicts where IDoFT's detection commit IS one FlakeBench used ===")
print(pd.crosstab(per["group"], per["same_commit"], margins=True).to_string())
print("\nStatus of same-commit conflicts:");  print(per[per["same_commit"]]["status"].value_counts().to_string())
print("\nConflicts per project (top 10):");   print(per["project"].value_counts().head(10).to_string())

# ---- (b) ID conflicts: candidates for the NonDex run ----
idc = h[h["Category"].astype(str).str.contains(r"(^|;)ID($|;)")]
print("\n=== (b) ID conflicts: NonDex candidates ===")
print(idc[["project", "Module Path", FQN, "SHA Detected", "Status", "same_commit"]]
      .drop_duplicates().sort_values(["same_commit", "project"], ascending=[False, True]).to_string(index=False))

### 8 Followup verification

In [1]:
%cd /content
!git clone -q https://github.com/UT-SE-Research/FlakyLens.git
!git clone -q --depth 1 https://github.com/TestingResearchIllinois/idoft.git
!ls /content

/content
FlakyLens  idoft  sample_data


### 8.1. Verify the trailing newline in the dataset and model input

In [2]:
import pandas as pd
pd.set_option("display.width", 250)
ds  = pd.read_csv("/content/FlakyLens/FlakeBench/FlakeBench_dataset.csv")
res = pd.read_csv("/content/FlakyLens/src/FlakyLens_Categorization_PerProject-result/Finetuned_Result_with_tokens.csv")

def ending(c):
    c = str(c)
    if c.endswith("\r\n"): return "ends \\r\\n"
    if c.endswith("\n"):   return "ends \\n"
    if c != c.rstrip():    return "ends other whitespace"
    return "no trailing whitespace"

for name, codes, flaky in [("FlakeBench_dataset.csv (full_code)", ds["full_code"], ds["category"] != 5),
                           ("results file (test_code = model input)", res["test_code"], res["Ground_Truth"] != 5)]:
    print(f"\n=== {name} ===")
    print(pd.crosstab(flaky.map({True: "flaky", False: "non-flaky"}), codes.map(ending)))


=== FlakeBench_dataset.csv (full_code) ===
full_code  ends \n  ends \r\n  no trailing whitespace
category                                             
flaky            0          0                     280
non-flaky     8149        145                       0

=== results file (test_code = model input) ===
test_code     ends \n  ends \r\n  no trailing whitespace
Ground_Truth                                            
flaky               0          0                     280
non-flaky        8149        145                       0


### 8.2. Inspect training data and preprocessing


In [3]:
%cd /content/FlakyLens/src
!echo "=== how the training script reads and tokenizes data ===" && grep -n -E "read_csv|batch_encode_plus|tokenizer\(|strip\(|replace\(|re\.sub" Bert_train_per_project.py | head -30
!echo "=== training split files ===" && ls FlakyLens_Categorization_PerProject-Data FlakyLens_Categorization_PerProject-Data/data_split

/content/FlakyLens/src
=== how the training script reads and tokenizes data ===
318:    df = pd.read_csv(dataset_path)  # Load the dataset
345:        #train_dataset = pd.read_csv(os.path.join(data_name_dir, train_file))
347:        '''train_data_Org = pd.read_csv(os.path.join(data_name_dir+"/data_split/with_50_percent_noisy_data", train_file))
348:        valid_data = pd.read_csv(os.path.join(data_name_dir+"/data_split/with_50_percent_noisy_data", valid_file))'''
350:        train_dataset = pd.read_csv(os.path.join(data_name_dir, train_file))
351:        test_dataset = pd.read_csv(os.path.join(data_name_dir, test_file))
359:            train_data_Org = pd.read_csv(os.path.join(data_name_dir+"/data_split", train_file))
360:            valid_data = pd.read_csv(os.path.join(data_name_dir+"/data_split", valid_file))
361:            test_dataset = pd.read_csv(os.path.join(data_name_dir, test_file))
510:    df = pd.read_csv(dataset_path)
=== training split files ===
FlakyLens_Categorization

In [4]:
import glob, pandas as pd
for f in sorted(glob.glob("/content/FlakyLens/src/FlakyLens_Categorization_PerProject-Data/data_split/train_set_*.csv")):
    d = pd.read_csv(f)
    code_col  = next((c for c in d.columns if "code" in c.lower()), d.columns[0])
    label_col = next((c for c in ["which_tests", "category", "label", "Ground_Truth"] if c in d.columns), None)
    print(f"\n=== {f.split('/')[-1]}  rows={len(d)}  code column='{code_col}'  label column='{label_col}' ===")
    print("columns:", list(d.columns))
    if label_col is not None:
        is_flaky = d[label_col].astype(str).isin(["0", "1", "2", "3", "4"])
        print(pd.crosstab(is_flaky.map({True: "flaky", False: "non-flaky"}), d[code_col].map(ending)))


=== train_set_1.csv  rows=5076  code column='full_code'  label column='category' ===
columns: ['id', 'project', 'test_name', 'full_code', 'label', 'category']
full_code  ends \n  ends \r\n  no trailing whitespace
category                                             
flaky            0          0                     172
non-flaky     4872         32                       0

=== train_set_2.csv  rows=5114  code column='full_code'  label column='category' ===
columns: ['id', 'project', 'test_name', 'full_code', 'label', 'category']
full_code  ends \n  ends \r\n  no trailing whitespace
category                                             
flaky            0          0                     142
non-flaky     4863        109                       0

=== train_set_3.csv  rows=5100  code column='full_code'  label column='category' ===
columns: ['id', 'project', 'test_name', 'full_code', 'label', 'category']
full_code  ends \n  ends \r\n  no trailing whitespace
category                          

In [5]:
%cd /content/FlakyLens/src
!grep -n -iE "tokeniz|encode" Bert_train_per_project.py | head -20
!echo "=== in utils.py / data_processing.py ===" && grep -n -E "def .*token|batch_encode_plus|tokenizer\(" utils.py data_processing.py | head -20

/content/FlakyLens/src
11:from transformers import AdamW, AutoTokenizer, AutoModel, AutoConfig
25:from data_processing import sampling, Tokenizer, TensorConverter, DataLoaderFactory, create_tensor_and_data_loader
341:        model_name, tokenizer, auto_model = codebert_model_define()
382:        #tokens_train, tokens_val, tokens_test = tokenize_data(train_x, valid_x, test_x)
383:        tokenizer_instance = Tokenizer(tokenizer)
386:        tokens_val = tokenizer_instance.tokenize_validation_data(valid_x)
387:        tokens_train = tokenizer_instance.tokenize_training_data(train_x)
388:        tokens_test = tokenizer_instance.tokenize_test_data(test_x)
=== in utils.py / data_processing.py ===
utils.py:196:def contains_english_letter(token):
utils.py:199:def filter_tokens(sorted_token_attributions):
utils.py:231:def predict(inputs, attention_mask, model, target_token):
utils.py:238:def clean_token(token):
utils.py:242:def interpret_with_ig_codellama(prompt, tokenizer, model, model_inputs

### 8.3. Verify folds 2–4 and control for truncation


In [6]:
from google.colab import drive; drive.mount("/content/drive")
!mkdir -p /content/FlakyLens/models && cp /content/drive/MyDrive/flakylens_models/*.pt /content/FlakyLens/models/ && ls -lh /content/FlakyLens/models

Mounted at /content/drive
total 1.9G
-rw------- 1 root root 478M Nov 23 14:13 per_project_model_weights_on__dataset_project_group_1.pt
-rw------- 1 root root 478M Nov 23 14:14 per_project_model_weights_on__dataset_project_group_2.pt
-rw------- 1 root root 478M Nov 23 14:14 per_project_model_weights_on__dataset_project_group_3.pt
-rw------- 1 root root 478M Nov 23 14:14 per_project_model_weights_on__dataset_project_group_4.pt
-rw-r--r-- 1 root root  133 Nov 23 14:10 README.md


In [7]:
import sys, gc, torch, numpy as np, pandas as pd
from transformers import AutoConfig, AutoTokenizer, AutoModel
sys.path.insert(0, "/content/FlakyLens/src")
from codebert_model import BERT_Arch
device, MODEL_NAME = "cuda", "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
names = {0:"async", 1:"concurrency", 2:"time", 3:"unordered", 4:"order-dep", 5:"non-flaky"}

def load_flakylens(group):
    config = AutoConfig.from_pretrained(MODEL_NAME, return_dict=False, output_hidden_states=True)
    model  = BERT_Arch(AutoModel.from_pretrained(MODEL_NAME, config=config), 6)
    sd = torch.load(f"/content/FlakyLens/models/per_project_model_weights_on__dataset_project_group_{group}.pt", map_location="cpu")
    r = model.load_state_dict(sd, strict=False)
    print(f"fold {group}: missing={r.missing_keys} unexpected={r.unexpected_keys}")
    return model.to(device).eval()

@torch.no_grad()
def predict(model, codes, batch_size=32):
    probs = []
    for i in range(0, len(codes), batch_size):
        enc = tokenizer(list(codes[i:i+batch_size]), max_length=512, padding="max_length", truncation=True, return_tensors="pt")
        probs.append(model(enc["input_ids"].to(device), enc["attention_mask"].to(device)).exp().cpu())
    probs = torch.cat(probs).numpy()
    return probs.argmax(1), probs
print("ready")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

ready


In [8]:
res = pd.read_csv("/content/FlakyLens/src/FlakyLens_Categorization_PerProject-result/Finetuned_Result_with_tokens.csv")
agree_rows, trunc_rows = [], []
for g in [1, 2, 3, 4]:
    m = load_flakylens(g)
    d = res[res["project_group"] == g].reset_index(drop=True)

    # (1) do our predictions match the authors' saved ones? (fold 1 was already 2,229/2,229)
    if g != 1:
        p = predict(m, d["test_code"].astype(str).tolist())[0]
        agree_rows.append({"fold": g, "tests": len(d), "agree": int((p == d["Prediction"].values).sum())})

    # (2) truncation control: long flaky tests, cut so the end is VISIBLE, with vs without a trailing newline
    fl = d[d["Ground_Truth"] != 5]
    long_codes, info = [], []
    for code, true, saved in zip(fl["test_code"].astype(str), fl["Ground_Truth"], fl["Prediction"]):
        enc = tokenizer(code, add_special_tokens=False, return_offsets_mapping=True)
        if len(enc["input_ids"]) <= 510: continue                    # same 61 '>512 tokens' tests as Step 16
        long_codes.append(code[:enc["offset_mapping"][499][1]])      # keep the first 500 tokens' worth of text
        info.append((names[true], saved))
    if long_codes:
        p_cut    = predict(m, long_codes)[0]
        p_cut_nl = predict(m, [c + "\n" for c in long_codes])[0]
        for (true, saved), a, b in zip(info, p_cut, p_cut_nl):
            trunc_rows.append({"fold": g, "true": true, "cut_same_as_saved": a == saved,
                               "cut_pred_nonflaky": a == 5, "flipped_with_newline": (a != 5) and (b == 5)})
    del m; gc.collect(); torch.cuda.empty_cache()

print("\n=== (1) agreement with the authors' saved predictions ===");  print(pd.DataFrame(agree_rows).to_string(index=False))
t = pd.DataFrame(trunc_rows)
print(f"\n=== (2) truncation control: {len(t)} long flaky tests, cut to ~500 tokens ===")
print("cut version predicted the same as the saved prediction:", int(t["cut_same_as_saved"].sum()), "of", len(t))
print("cut version already predicted non-flaky:", int(t["cut_pred_nonflaky"].sum()))
print("FLIPPED to non-flaky when a newline is added at the (now visible) end:", int(t["flipped_with_newline"].sum()), "of", len(t))
print(t.groupby("true")["flipped_with_newline"].agg(n="size", flipped="sum").to_string())
t.to_csv("/content/drive/MyDrive/flakylens_results/step2_truncation_control.csv", index=False)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fold 1: missing=[] unexpected=[]


... (3 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fold 3: missing=[] unexpected=[]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fold 4: missing=[] unexpected=[]

=== (1) agreement with the authors' saved predictions ===
 fold  tests  agree
    2   2181   2181
    3   2199   2199
    4   1965   1965

=== (2) truncation control: 61 long flaky tests, cut to ~500 tokens ===
cut version predicted the same as the saved prediction: 60 of 61
cut version already predicted non-flaky: 0
FLIPPED to non-flaky when a newline is added at the (now visible) end: 3 of 61
              n  flipped
true                    
async        31        0
concurrency  14        0
order-dep     5        2
time          6        0
unordered     5        1


### 8.4. Trace the training input from file loading to tokenization

In [9]:
%cd /content/FlakyLens/src
!echo "=== Bert_train_per_project.py, lines 350-390 ===" && sed -n '350,390p' Bert_train_per_project.py
!echo "=== data_processing.py, lines 437-450 ===" && sed -n '437,450p' data_processing.py

/content/FlakyLens/src
=== Bert_train_per_project.py, lines 350-390 ===
        train_dataset = pd.read_csv(os.path.join(data_name_dir, train_file))
        test_dataset = pd.read_csv(os.path.join(data_name_dir, test_file))
        if need_valid_set_create: # If True
            train_x, train_y, valid_x, valid_y = generate_train_valid_data_split(data_name_dir, train_dataset, test_dataset, project_group, y)
            # Fix: select only the text column as Series
            train_x = train_x['full_code']
            valid_x = valid_x['full_code']
        else: # If False
            # When I already have the saved data
            train_data_Org = pd.read_csv(os.path.join(data_name_dir+"/data_split", train_file))
            valid_data = pd.read_csv(os.path.join(data_name_dir+"/data_split", valid_file))
            test_dataset = pd.read_csv(os.path.join(data_name_dir, test_file))
            valid_x = valid_data.drop(columns=['category'])  # Keep all columns except 'category'
       

### 8.5. Verify that the Mapper test matches the FlakeBench example

In [10]:
%cd /content
![ -d Mapper ] || git clone -q https://github.com/abel533/Mapper.git
!cd /content/Mapper && git checkout -q 1764748eedb2f320a0d1c43cb4f928c4ccb1f2f5 && git log -1 --oneline
!echo "=== every ComplexEntityTest.java in the repo ===" && find /content/Mapper -name ComplexEntityTest.java

/content
1764748 (HEAD) 添加日志输出异常和警告信息，fixed #IXNLU
=== every ComplexEntityTest.java in the repo ===
/content/Mapper/core/src/test/java/tk/mybatis/mapper/mapperhelper/ComplexEntityTest.java


In [11]:
import pandas as pd, re
ds = pd.read_csv("/content/FlakyLens/FlakeBench/FlakeBench_dataset.csv")
rows = ds[(ds["project"].str.lower() == "abel533_mapper") & ds["test_name"].str.contains("ComplexEntityTest", na=False)]
print("FlakeBench rows:"); print(rows[["test_name", "label"]].to_string(index=False))

def method_block(src, name="test"):
    """Return the text of 'public void <name>(' including its @Test line, up to the matching closing brace."""
    i = src.find(f"public void {name}(")
    start = src.rfind("@Test", 0, i)
    j, depth = src.find("{", i), 0
    for k in range(j, len(src)):
        depth += src[k] == "{"; depth -= src[k] == "}"
        if depth == 0: return src[start:k+1]

norm = lambda s: re.sub(r"\s+", "", s)              # ignore all whitespace/formatting differences
src  = open("/content/Mapper/core/src/test/java/tk/mybatis/mapper/mapperhelper/ComplexEntityTest.java").read()
repo_method = method_block(src)
for _, r in rows.iterrows():
    same = norm(r["full_code"]) == norm(repo_method)
    print(f"\n{r['test_name']}: identical to core/ test method (ignoring whitespace)? {same}")
    if not same:
        print("--- FlakeBench ---\n", r["full_code"][:800], "\n--- repo (core/) ---\n", repo_method[:800])

FlakeBench rows:
             test_name     label
ComplexEntityTest.test non-flaky

ComplexEntityTest.test: identical to core/ test method (ignoring whitespace)? True


### 8.6. Inspect the `apply_smote` implementation

In [12]:
%cd /content/FlakyLens/src
!grep -n -A12 "def apply_smote" *.py | head -30

/content/FlakyLens/src
Bert_train_per_project.py:230:def apply_smote(train_data_Org, train_y_Org):
Bert_train_per_project.py-231-    from imblearn.over_sampling import SMOTE
Bert_train_per_project.py-232-    from sklearn.feature_extraction.text import TfidfVectorizer
Bert_train_per_project.py-233-
Bert_train_per_project.py-234-    # Step 1: Store Original Text Data
Bert_train_per_project.py-235-    train_x_Org = train_data_Org['full_code']  # Original text
Bert_train_per_project.py-236-    train_y_Org = train_data_Org['category']   # Original labels
Bert_train_per_project.py-237-
Bert_train_per_project.py-238-    # Skip SMOTE, return original data in required format
Bert_train_per_project.py-239-    train_x = train_x_Org.reset_index(drop=True)
Bert_train_per_project.py-240-    train_y = train_y_Org.reset_index(drop=True)
Bert_train_per_project.py-241-    train_data = pd.DataFrame({"full_code": train_x, "category": train_y})
Bert_train_per_project.py-242-    return train_x, train_data, 

### 8.7. Reconstruct IDoFT conflicts for manual review

In [14]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
ds    = pd.read_csv("/content/FlakyLens/FlakeBench/FlakeBench_dataset.csv")
idoft = pd.read_csv("/content/idoft/pr-data.csv")
FQN   = "Fully-Qualified Test Name (packageName.ClassName.methodName)"
key   = lambda s: str(s).lower().replace("/", "_")

# IDoFT keys (same as Step 19)
idoft["proj"] = idoft["Project URL"].astype(str).str.rstrip("/").str.split("github.com/").str[-1].map(key)
nm = idoft[FQN].astype(str).str.replace(r"\[.*\]$", "", regex=True)
idoft["cls_method"] = nm.str.split(".").str[-2:].str.join(".")

# commits FlakeBench used, per project
m  = pd.read_csv("/content/FlakyLens/FlakeBench/filtered_tests_with_owner_sha.csv", header=None, names=["repo", "shas"])
m["proj"] = m["repo"].map(key)
fb_shas = m.assign(sha=m["shas"].astype(str).str.split(";")).explode("sha").groupby("proj")["sha"].apply(set).to_dict()

# FlakeBench NON-flaky tests matched to IDoFT by project + Class.method
nf = ds[ds["category"] == 5].copy()
nf["proj"] = nf["project"].str.lower()
nf["cls_method"] = nf["test_name"].astype(str).str.replace(r"(\.\d+)+\.?$", "", regex=True).str.split(".").str[-2:].str.join(".")
conf = nf.merge(idoft[["proj", "cls_method", "SHA Detected", "Module Path", FQN, "Category", "Status", "PR Link"]],
                on=["proj", "cls_method"], how="inner")
conf["same_commit"] = [str(s) in fb_shas.get(p, set()) for p, s in zip(conf["proj"], conf["SHA Detected"])]

n_tests = conf[["project", "test_name"]].drop_duplicates().shape[0]
amb = conf.groupby(["project", "test_name"])[FQN].nunique()
print(f"Conflicting non-flaky tests: {n_tests} (expected 129) | IDoFT rows: {len(conf)}")
print(f"Name-matching risk: {int((amb > 1).sum())} tests match MORE THAN ONE different IDoFT test (different package/module)")
conf.drop(columns=["full_code"]).to_csv("/content/drive/MyDrive/flakylens_results/step19_conflicts.csv", index=False)
print("Saved conflicts table to Drive.")

# five cases to check by hand
picks = [("apache_hadoop", "TestRpcProgramNfs3.testGetattr"), ("apache_hadoop", "TestReaddir.testReaddirBasic"),
         ("NationalSecurityAgency_timely", "AuthCacheTest.testGetAuths")]
picks += [("wildfly_wildfly", t) for t in conf[conf["project"] == "wildfly_wildfly"]["test_name"].drop_duplicates().head(2)]
for p, t in picks:
    rows = conf[(conf["project"] == p) & (conf["test_name"] == t)]
    if rows.empty: print(f"\n### {p} | {t}: not found"); continue
    r = rows.iloc[0]
    print(f"\n### {p} | {t}")
    print(rows[["Module Path", FQN, "SHA Detected", "same_commit", "Category", "Status", "PR Link"]].to_string(index=False))
    print("--- FlakeBench code (first 15 lines) ---\n" + "\n".join(str(r["full_code"]).splitlines()[:15]))

Conflicting non-flaky tests: 129 (expected 129) | IDoFT rows: 134
Name-matching risk: 5 tests match MORE THAN ONE different IDoFT test (different package/module)
Saved conflicts table to Drive.

### apache_hadoop | TestRpcProgramNfs3.testGetattr
                        Module Path   Fully-Qualified Test Name (packageName.ClassName.methodName)                             SHA Detected  same_commit Category   Status                                    PR Link
hadoop-hdfs-project/hadoop-hdfs-nfs org.apache.hadoop.hdfs.nfs.nfs3.TestRpcProgramNfs3.testGetattr cc2babc1f75c93bf89a8f10da525f944c15d02ea        False   OD-Vic Accepted https://github.com/apache/hadoop/pull/6329
--- FlakeBench code (first 15 lines) ---
  @Test(timeout = 60000)
  public void testGetattr() throws Exception {
    HdfsFileStatus status = nn.getRpcServer().getFileInfo("/tmp/bar");
    long dirId = status.getFileId();
    int namenodeId = Nfs3Utils.getNamenodeId(config);
    FileHandle handle = new FileHandle(dirId, namen

### 8.8. Check blank lines and comment lines across training and validation splits

In [1]:
!cd /content && [ -d FlakyLens ] || git clone -q https://github.com/UT-SE-Research/FlakyLens.git
import glob, pandas as pd, re
def feats(c):
    c = str(c); core = c.strip()
    return pd.Series({"ends_newline": c.endswith("\n"),
                      "has_blank_line": any(not l.strip() for l in core.split("\n")),
                      "has_comment_line": bool(re.search(r"(?m)^\s*(//|/\*|\*)", core))})
for f in sorted(glob.glob("/content/FlakyLens/src/FlakyLens_Categorization_PerProject-Data/data_split/*_set_*.csv")):
    d = pd.read_csv(f)
    kind = (d["category"] != 5).map({True: "flaky", False: "non-flaky"})
    s = d["full_code"].apply(feats).groupby(kind).sum().astype(int)
    s["n"] = kind.value_counts()
    print(f"\n=== {f.split('/')[-1]} ===\n{s.to_string()}")


=== train_set_1.csv ===
           ends_newline  has_blank_line  has_comment_line     n
category                                                       
flaky                 0               0                 0   172
non-flaky          4904            2524              1175  4904

=== train_set_2.csv ===
           ends_newline  has_blank_line  has_comment_line     n
category                                                       
flaky                 0               0                 0   142
non-flaky          4972            2600              1226  4972

=== train_set_3.csv ===
           ends_newline  has_blank_line  has_comment_line     n
category                                                       
flaky                 0               0                 0   178
non-flaky          4922            2690              1179  4922

=== train_set_4.csv ===
           ends_newline  has_blank_line  has_comment_line     n
category                                                       
flak